In [0]:
client_id = dbutils.secrets.get("kv-scope", "sp-client-id")
client_secret = dbutils.secrets.get("kv-scope", "sp-secret-value")
tenant_id = dbutils.secrets.get("kv-scope", "sp-tenant-id")

In [0]:
storage_account = "adlssalesprojectkiran"
tenant_id = dbutils.secrets.get("kv-scope", "sp-tenant-id")
client_id = dbutils.secrets.get("kv-scope", "sp-client-id")
client_secret = dbutils.secrets.get("kv-scope", "sp-secret-value")

spark.conf.set(f"fs.azure.account.auth.type.{storage_account}.dfs.core.windows.net", "OAuth")
spark.conf.set(f"fs.azure.account.oauth.provider.type.{storage_account}.dfs.core.windows.net",
               "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
spark.conf.set(f"fs.azure.account.oauth2.client.id.{storage_account}.dfs.core.windows.net", client_id)
spark.conf.set(f"fs.azure.account.oauth2.client.secret.{storage_account}.dfs.core.windows.net", client_secret)
spark.conf.set(f"fs.azure.account.oauth2.client.endpoint.{storage_account}.dfs.core.windows.net",
               f"https://login.microsoftonline.com/{tenant_id}/oauth2/token")


In [0]:
#-- ADF PARAMETERS

dbutils.widgets.text("p_table_name", "")
dbutils.widgets.text("p_delimiter", "")

table_name = dbutils.widgets.get("p_table_name")
delimiter  = dbutils.widgets.get("p_delimiter")

print("Processing table:", table_name)
#print("Source type:", source_type)

In [0]:
#-- IMPORTS

from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, col, current_timestamp, lit
from pyspark.sql.types import *
import re

In [0]:
#-- TABLE CONFIG
table_config = [
    ("ORDER_HEADER", "SQLServer", "ORDER_NUMBER"),
    ("ORDER_DETAILS", "SQLServer", "ORDER_DETAIL_CODE"),
    ("PRODUCT", "SQLServer", "PRODUCT_NUMBER"),
    ("RETURNED_ITEM", "SQLServer", "RETURN_CODE"),
    ("INVENTORY_LEVELS", "SQLServer", ["INVENTORY_YEAR","INVENTORY_MONTH","WAREHOUSE_BRANCH_CODE","PRODUCT_NUMBER"]),
    ("BRANCH", "SQLServer", "BRANCH_CODE"),
    ("ORDER_METHOD", "Flat_File", "ORDER_METHOD_CODE"),
    ("RETURN_REASON", "Flat_File", "RETURN_REASON_CODE"),
    ("COUNTRY", "Flat_File", "COUNTRY_CODE"),
    ("WAREHOUSE", "Flat_File", "BRANCH_CODE"),
    ("Retailer", "Flat_File", "RETAILER_SITE_CODE"),
    ("PRODUCT_NAME_LOOKUP", "Flat_File", "PRODUCT_NUMBER")
]

In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType, TimestampType

schema_dict = {

    "ORDER_HEADER" : StructType([
         StructField("ORDER_NUMBER", IntegerType(), True),
         StructField("RETAILER_NAME", StringType(), True),
         StructField("RETAILER_NAME_MB", StringType(), True),
         StructField("RETAILER_SITE_CODE", IntegerType(), True),
         StructField("RETAILER_CONTACT_CODE", IntegerType(), True),
         StructField("RAW_STAFF_CODE", IntegerType(), True),
         StructField("RAW_BRANCH_CODE", IntegerType(), True),
         StructField("ORDER_DATE", TimestampType(), True),
         StructField("ORDER_CLOSE_DATE", TimestampType(), True),
         StructField("ORDER_METHOD_CODE", IntegerType(), True)
         ]),
    "PRODUCT" : StructType([
        StructField("PRODUCT_NUMBER", IntegerType(), True), 
        StructField("BASE_PRODUCT_NUMBER", IntegerType(), True), 
        StructField("INTRODUCTION_DATE", TimestampType(), True), 
        StructField("DISCONTINUED_DATE", TimestampType(), True),
        StructField("PRODUCT_TYPE_CODE", IntegerType(), True),
        StructField("PRODUCT_COLOR_CODE", IntegerType(), True),
        StructField("PRODUCT_SIZE_CODE", IntegerType(), True),
        StructField("PRODUCT_BRAND_CODE", IntegerType(), True),
        StructField("PRODUCTION_COST", DoubleType(), True),
        StructField("GROSS_MARGIN", DoubleType(), True),
        StructField("PRODUCT_IMAGE", StringType(), True)
        ]),
     "ORDER_DETAILS" : StructType([
        StructField("ORDER_DETAIL_CODE", IntegerType(), True), 
        StructField("ORDER_NUMBER", IntegerType(), True), 
        StructField("SHIP_DATE", TimestampType(), True), 
        StructField("PRODUCT_NUMBER", IntegerType(), True), 
        StructField("PROMOTION_CODE", IntegerType(), True), 
        StructField("QUANTITY", IntegerType(), True), 
        StructField("UNIT_COST",DoubleType(),True),
        StructField("UNIT_PRICE",DoubleType(),True),
        StructField("UNIT_SALE_PRICE",DoubleType(),True)
        ]),
     "RETURNED_ITEM" : StructType([
        StructField("RETURN_CODE",IntegerType(),True),
        StructField("RETURN_DATE",TimestampType(),True),
        StructField("ORDER_DETAIL_CODE",IntegerType(),True),
        StructField("RETURN_REASON_CODE",IntegerType(),True),
        StructField("RETURN_QUANTITY",IntegerType(),True),
        StructField("ASSIGNED_TO",IntegerType(),True),
        StructField("FOLLOW_UP_CODE",IntegerType(),True),
        StructField("COMMENTS",StringType(),True),
        StructField("DATE_ADVISED",TimestampType(),True)
        ]),
     "INVENTORY_LEVELS" : StructType([
       StructField("INVENTORY_YEAR",IntegerType(),True),
       StructField("INVENTORY_MONTH",IntegerType(),True),
       StructField("WAREHOUSE_BRANCH_CODE",IntegerType(),True),
       StructField("PRODUCT_NUMBER",IntegerType(),True),
       StructField("OPENING_INVENTORY",IntegerType(),True),
       StructField("QUANTITY_SHIPPED",IntegerType(),True), 
       StructField("ADDITIONS",IntegerType(),True), 
       StructField("UNIT_COST",DoubleType(),True), 
       StructField("CLOSING_INVENTORY",IntegerType(),True), 
       StructField("AVERAGE_UNIT_COST",DoubleType(),True)
         ]),
     "BRANCH": StructType([
       StructField("BRANCH_CODE", IntegerType(), True),
       StructField("ADDRESS1", StringType(), True),
       StructField("CITY", StringType(), True),
       StructField("COUNTRY_CODE", StringType(), True),
       StructField("WAREHOUSE_BRANCH_CODE", StringType(), True),
    ]),

    "ORDER_METHOD": StructType([
        StructField("ORDER_METHOD_CODE", IntegerType(), True),
        StructField("ORDER_METHOD_EN", StringType(), True)
    ]),

    "RETURN_REASON": StructType([
        StructField("RETURN_REASON_CODE", IntegerType(), True),
        StructField("REASON_DESCRIPTION_EN", StringType(), True)
    ]),

    "COUNTRY": StructType([
        StructField("COUNTRY_CODE", IntegerType(), True),
        StructField("COUNTRY_ID", StringType(), True),
        StructField("COUNTRY_NAME", StringType(), True)
    ]),

    "WAREHOUSE": StructType([
        StructField("BRANCH_CODE", IntegerType(), True),
        StructField("ADDRESS1", StringType(), True),
        StructField("CITY", StringType(), True),
        StructField("POSTAL_ZONE", StringType(), True),
        StructField("COUNTRY_CODE", StringType(), True),
        StructField("WAREHOUSE_BRANCH_CODE", StringType(), True)
    ]),

    "Retailer": StructType([
        StructField("RETAILER_NAME", StringType(), True),
        StructField("RETAILER_SITE_CODE", IntegerType(), True),
        StructField("RETAILER_CONTACT_CODE", IntegerType(), True)
    ]),

    "PRODUCT_NAME_LOOKUP": StructType([
        StructField("PRODUCT_NUMBER", IntegerType(), True),
        StructField("PRODUCT_LANGUAGE", StringType(), True),
        StructField("PRODUCT_NAME", StringType(), True),
        StructField("PRODUCT_DESCRIPTION", StringType(), True) 
    ])
}       


In [0]:
#-- HELPER FUNCTIONS

rename_config = {
    "ORDER_METHOD": {
        "ORDER_METHOD_CODE": "METHOD_CODE",
        "ORDER_METHOD_EN": "METHOD_NAME"
    },
    "RETURN_REASON": {
        "REASON_DESCRIPTION_EN": "RETURN_REASON_DESC"
    },
    "WAREHOUSE": {
        "ADDRESS1": "ADDRESS",
        "POSTAL_ZONE": "POSTAL_CODE"
    }
}

def apply_rename_column(df, table):
    if table in rename_config:
        for old_col, new_col in rename_config[table].items():
            df = df.withColumnRenamed(old_col, new_col)
    return df

def add_audit_columns(df, source):
    return (df
        .withColumn("SOURCE_ID", lit(source))
        .withColumn("DataDate", current_timestamp())
        .withColumn("UpdateDate", current_timestamp())
    )
    
def get_latest_file(path):
    files = dbutils.fs.ls(path)
    latest_file = sorted(files, key=lambda x: x.name, reverse=True)[0].path
    return latest_file

def truncate_table(table):

    jdbc_password = dbutils.secrets.get("kv-scope", "secret-sql-AdminSalespwd")
    jdbc_url = "jdbc:sqlserver://admin-salesproject.database.windows.net:1433;database=ASQL_SalesProject_Kiran;encrypt=true;trustServerCertificate=false;hostNameInCertificate=*.database.windows.net;loginTimeout=30;"

    spark._sc._gateway.jvm.java.sql.DriverManager.getConnection(
        jdbc_url,
        "Admin_Sales",
        jdbc_password
    ).createStatement().execute(f"TRUNCATE TABLE ingestion.{table}")

    print(f"Truncated ingestion.{table}")

In [0]:
#-- MAIN PROCESSING

for table, source, pk in table_config:

    #- Only process table sent from ADF
    if table != table_name:
        continue

    print(f"Processing: {table}")

    base_path = "SQLServer/Target" if source == "SQLServer" else "Flat_File/Target"
    reject_path = base_path.replace("Target", "Reject")

    path = f"abfss://raw@adlssalesprojectkiran.dfs.core.windows.net/{base_path}/{table}/"

    # READ LATEST RAW FILE

    latest_file = get_latest_file(path)
    print("Latest file:", latest_file)

    df = spark.read.format("csv") \
        .option("header", "true") \
        .option("delimiter", delimiter) \
        .option("quote", "\"") \
        .option("escape", "\"") \
        .option("multiLine", "true") \
        .option("encoding", "UTF-8") \
        .schema(schema_dict.get(table)) \
        .load(latest_file)

    # PRIMARY KEY VALIDATION

    if isinstance(pk, list):
        window = Window.partitionBy(*pk).orderBy(*pk)
        condition = " AND ".join([f"{c} IS NOT NULL" for c in pk])
    else:
        window = Window.partitionBy(pk).orderBy(pk)
        condition = f"{pk} IS NOT NULL"

    df2 = df.withColumn("rn", row_number().over(window))

    df_valid = df2.filter(f"rn = 1 AND {condition}").drop("rn")
    df_reject = df2.filter(f"rn > 1 OR NOT ({condition})").drop("rn")

    # RENAME + AUDIT

    df_valid = apply_rename_column(df_valid, table)
    df_valid = add_audit_columns(df_valid, source)

    # WRITE REJECT

    df_reject.write.mode("append").parquet(
        f"abfss://raw@adlssalesprojectkiran.dfs.core.windows.net/{reject_path}/{table}/"
    )

    print("Reject written")
    print(df_reject.count())


    # TRUNCATE + LOAD (FULL LOAD SUPPORT)

    truncate_table(table)

    jdbc_password = dbutils.secrets.get("kv-scope", "secret-sql-AdminSalespwd")

    jdbc_url = "jdbc:sqlserver://admin-salesproject.database.windows.net:1433;database=ASQL_SalesProject_Kiran;encrypt=true;trustServerCertificate=false;hostNameInCertificate=*.database.windows.net;loginTimeout=30;"

    df_valid.write \
        .format("jdbc") \
        .option("url", jdbc_url) \
        .option("dbtable", f"ingestion.{table}") \
        .option("user", "Admin_Sales") \
        .option("password", jdbc_password) \
        .mode("append") \
        .save()

    print("Valid written")
    print(df_valid.count())

    print(f"Completed: {table}")


print("Notebook Execution Finished")